<a href="https://colab.research.google.com/github/yagu6173/3238-NLP-Final-Project/blob/main/NLP_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers
!pip install evaluate
!pip install rouge_score
!pip install datasets

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=cde0a7c364025b4ef4a142bc9a5d098a53962904ee5bd6befa9fa2f727617d5e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
from datasets import load_dataset
dataset = load_dataset("abisee/cnn_dailymail","3.0.0")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "facebook/bart-large-cnn" #"t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [5]:
subset = dataset['train'].select(range(3))
for item in subset:
  inputs = tokenizer(item['article'], truncation=True, return_tensors="pt")
  summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        early_stopping=True
    )
  summary = tokenizer.decode(summary_ids)
  print("machine summary: ",summary)
  print("human summary", item['highlights'])

machine summary:  ["</s><s>Harry Potter star Daniel Radcliffe turns 18 on Monday. He gains access to a reported £20 million ($41.1 million) fortune. Radcliffe's earnings from the first five Potter films have been held in a trust fund. Details of how he'll mark his landmark birthday are under wraps.</s>"]
human summary Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .
machine summary:  ['</s><s>Judge Steven Leifman is an advocate for justice and the mentally ill. About one-third of all people in Miami-Dade county jails are mentally ill, he says. He says the sheer volume is overwhelming the system. Starting in 2008, many inmates will be sent to a new mental health facility.</s>']
human summary Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidabl

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

machine_summaries = []
human_summaries = []

# Re-generate and collect summaries for the first 3 items
subset = dataset['train'].select(range(3))
for item in subset:
  inputs = tokenizer(item['article'], truncation=True, return_tensors="pt")
  summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        early_stopping=True
    )
  # Decode with skip_special_tokens for clean text suitable for ROUGE
  machine_summaries.append(tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True))
  human_summaries.append(item['highlights'])

# Compute ROUGE scores
results = rouge.compute(predictions=machine_summaries, references=human_summaries)



In [36]:
print("\nROUGE Scores:")
for key, value in results.items():
  print(key, round(value,4))


ROUGE Scores:
rouge1 0.4832
rouge2 0.2457
rougeL 0.3078
rougeLsum 0.4215
